In [19]:
import torch
import torchvision
import torchvision.transforms as transforms
from scipy.stats import levy_stable

In [ ]:
import torchvision
import torchvision.transforms as transforms

def get_mnist_data():
    # 1. Define transformation to flatten the 28x28 images into a 784 vector
    # and normalize pixel values to [0, 1]
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: torch.flatten(x))
    ])
    
    # 2. Download Training and Testing sets
    train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    
    # 3. Extract tensors directly for your full-batch experiment
    X_train = train_set.data.float() / 255.0
    X_train = X_train.view(-1, 28*28) # Shape: [60000, 784]
    y_train = train_set.targets        # Shape: [60000] (LongTensor for integers 0-9)
    
    X_test = test_set.data.float() / 255.0
    X_test = X_test.view(-1, 28*28)   # Shape: [10000, 784]
    y_test = test_set.targets          # Shape: [10000]
    
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = get_mnist_data()

In [9]:
import torch.nn as nn

class MNISTModel(nn.Module):
    def __init__(self):
        super(MNISTModel, self).__init__()
        # Change input from 123 to 784; output from 1 to 10
        self.linear = nn.Linear(784, 10)
        
    def forward(self, x):
        return self.linear(x)

# model_sgd = MNISTModel()
# model_signsgd = MNISTModel()

model_mnist = MNISTModel()

In [10]:
# 2. (DEFINING THE MODEL) & NON-CONVEX LOSS
# class NonConvexLogisticRegression(nn.Module):
#     def __init__(self, input_dim):
#         super(NonConvexLogisticRegression, self).__init__()
#         self.linear = nn.Linear(input_dim, 1, bias=False)
        
#     def forward(self, X_train):
#         return self.linear(X_train).squeeze()

def non_convex_regularizer(model, lam=0.1, alpha=1.0):
    reg = 0
    for param in model.parameters():
        # Geman-McClure: sum( (lam * w^2) / (1 + alpha * w^2) )
        reg += torch.sum((lam * param**2) / (1 + alpha * param**2))
    return reg

In [11]:
# 3. NOISE GENERATION (Lévy / Heavy-Tailed)
def add_heavy_tailed_noise(grad, alpha_levy=1.5, scale=0.01):
    """
    alpha_levy: 2.0 is Gaussian, 1.0 is Cauchy (Heavier).
    The paper suggests alpha < 2 captures the 'spikes'.
    """
    noise = levy_stable.rvs(alpha_levy, 0, scale=scale, size=grad.shape)
    return grad + torch.from_numpy(noise).float()

In [15]:
# def calculate_accuracy(outputs, targets):
#     # outputs shape: [Batch, 10]
#     # Find the index of the highest logit for each sample
#     _, predictions = torch.max(outputs, dim=1)
#     correct = (predictions == targets).sum().item()
#     return (correct / len(targets)) * 100.0

def calculate_accuracy(outputs, targets):
    # Get the index of the max logit along the class dimension (dim=1)
    predictions = torch.argmax(outputs, dim=1)
    correct = (predictions == targets).sum().item()
    return (correct / len(targets)) * 100.0

In [ ]:
# 4. TRAINING LOOP
def run_experiment(opt_type='sgd', alpha_levy=1.5, lr=0.01, epochs=100):
    # model = NonConvexLogisticRegression(123)

    # model_sgd = MNISTModel()      # If using the MNISTModel class from the previous step
    # model_signsgd = MNISTModel()    

    model = model_mnist

    criterion = nn.CrossEntropyLoss()
    losses = []
    
    for epoch in range(epochs):
        # Full batch gradient for stability control, then add synthetic noise
        outputs = model(X_train)

        #regularizer
        loss_val = criterion(outputs, y_train.long()) + non_convex_regularizer(model)

        #without regularizer
        # loss_val = criterion(outputs, y_train)
    
        
        model.zero_grad()
        loss_val.backward()
        
        with torch.no_grad():
            for param in model.parameters():
                # Inject Heavy-Tailed Noise into the gradient
                noisy_grad = add_heavy_tailed_noise(param.grad, alpha_levy)
                
                if opt_type == 'sgd':
                    param -= lr * noisy_grad
                elif opt_type == 'signsgd':
                    # DSignSGD Step: sign(grad + noise)
                    param -= lr * torch.sign(noisy_grad)
        
        losses.append(loss_val.item())

        # Inside your test loop:
        with torch.no_grad():
            test_outputs = model_sgd(X_test)
            test_acc = calculate_accuracy(test_outputs, y_test)
            print(f"Test Accuracy ({opt_type}): {test_acc:.2f}%")
        
    return losses, model



In [22]:
print("Running Experiments...")
history_sgd, model_sgd = run_experiment(opt_type='sgd', alpha_levy=2, lr=0.01, epochs = 10)
history_signsgd, model_signsgd = run_experiment(opt_type='signsgd', alpha_levy=2, lr=0.01, epochs = 10)

Running Experiments...
Test Accuracy: 80.91%
Test Accuracy: 80.96%
Test Accuracy: 81.14%
Test Accuracy: 81.25%
Test Accuracy: 81.38%
Test Accuracy: 81.49%
Test Accuracy: 81.58%
Test Accuracy: 81.69%
Test Accuracy: 81.84%
Test Accuracy: 81.94%
Test Accuracy: 83.46%
Test Accuracy: 81.11%
Test Accuracy: 83.89%
Test Accuracy: 81.49%
Test Accuracy: 83.69%
Test Accuracy: 80.88%
Test Accuracy: 82.96%
Test Accuracy: 81.88%
Test Accuracy: 80.72%
Test Accuracy: 81.39%
